# ASH-KV: Isolated INT8 Kernel Benchmark
This notebook isolates the OpenAI Triton INT8 compression kernels from the main SGLang engine. 
You can run this on any GPU (T4, A100, MI300X) to visually benchmark memory bandwidth and tune configurations via `@triton.autotune`.

In [ ]:
!pip install pandas matplotlib


In [ ]:
import torch
import triton
import triton.language as tl

# ---------------------------------------------------------------------------
# 1. TRITON KERNELS
# ---------------------------------------------------------------------------

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 128}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 512}, num_warps=8),
    ],
    key=['num_elements'],
)
@triton.jit
def encode_int8_kernel(x_ptr, out_ptr, scale_ptr, num_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < num_elements
    
    x = tl.load(x_ptr + offsets, mask=mask)
    abs_x = tl.abs(x)
    max_val = tl.max(abs_x)
    
    # Protect against zero division
    safe_max = tl.where(max_val == 0.0, 1.0, max_val)
    scale = safe_max / 127.0
    
    # Quantize to int8
    x_quant = tl.math.round(x / scale)
    x_quant = tl.math.clamp(x_quant, -128, 127)
    
    tl.store(out_ptr + offsets, x_quant.to(tl.int8), mask=mask)
    # A simplified autotune kernel storing one scale per block for benchmarking
    tl.store(scale_ptr + pid, scale)

def compress_int8(x):
    x = x.contiguous()
    num_elements = x.numel()
    out = torch.empty_like(x, dtype=torch.int8)
    
    # For benchmarking, we assume 1D blocks
    grid = lambda meta: (triton.cdiv(num_elements, meta['BLOCK_SIZE']),)
    
    # Allocate scale buffer (we estimate max grid size as num_elements // 128 + 1)
    scales = torch.empty(num_elements // 128 + 1, dtype=x.dtype, device=x.device)
    encode_int8_kernel[grid](x, out, scales, num_elements)
    return out, scales


In [ ]:
# ---------------------------------------------------------------------------
# 2. PERFORMANCE BENCHMARKING (GB/s)
# ---------------------------------------------------------------------------

@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],  # Argument name to use as an x-axis for the plot.
        x_vals=[2**i for i in range(12, 28, 1)],  # Different tensor sizes (from 4K to 134M elements)
        x_log=True,  # x axis is logarithmic.
        line_arg='provider',  # Argument name whose value corresponds to a different line in the plot.
        line_vals=['triton_int8'],  # Label name for the lines.
        line_names=['Triton INT8 Compression'],  # Legend names.
        styles=[('blue', '-')],  # Line styles.
        ylabel='GB/s',  # Label for the y-axis.
        plot_name='int8-compression-performance',  # Name for the plot.
        args={},  # Values for function arguments not in `x_names` and `y_name`.
    )
)
def benchmark(size, provider):
    x = torch.randn(size, device='cuda', dtype=torch.float16)
    quantiles = [0.5, 0.2, 0.8]
    
    if provider == 'triton_int8':
        # Benchmark the Triton kernel
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: compress_int8(x), quantiles=quantiles)
    
    # Calculate Memory Bandwidth (GB/s)
    # We read FP16 (2 bytes) and write INT8 (1 byte) = 3 bytes per element
    gbps = lambda ms: 3 * x.numel() / (ms * 1e6)
    return gbps(ms), gbps(max_ms), gbps(min_ms)

print("Starting Triton INT8 Benchmark...")
benchmark.run(print_data=True, show_plots=True)
